# GenAI Pipeline - Stage 2: Documentation Generation

**Stage 2 of 2**: Recursive documentation generation for source code repositories.

**Requirements**:
- GPU Runtime (T4 recommended)
- Stage 1 outputs uploaded to Google Drive
- Sufficient Colab quota

**Execution Time**: ~30-60 minutes depending on codebase size

---

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to Runtime > Change runtime type and select GPU (T4)")

In [ ]:
# Install/update required packages
!pip install -q --upgrade transformers accelerate torch pydantic networkx tqdm jinja2 libcst

# Optional: 4-bit quantization for memory efficiency
# !pip install -q bitsandbytes

print("✓ Packages installed")

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
print("✓ Google Drive mounted")

# Define paths
DRIVE_ROOT = Path('/content/drive/My Drive')
PIPELINE_ROOT = DRIVE_ROOT / 'GenAI_Pipeline_Automation'
INPUTS_DIR = PIPELINE_ROOT / 'stage2_inputs'
OUTPUTS_DIR = PIPELINE_ROOT / 'stage2_outputs'

# Create output directory
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Inputs dir: {INPUTS_DIR}")
print(f"Outputs dir: {OUTPUTS_DIR}")
print(f"Inputs exist: {INPUTS_DIR.exists()}")

if INPUTS_DIR.exists():
    print(f"\nContents of {INPUTS_DIR}:")
    for item in INPUTS_DIR.iterdir():
        print(f"  - {item.name}")

## Step 2: Clone Repository and Load Stage 1 Outputs

In [ ]:
# Clone the GenAI project from GitHub (the one you're documenting)
# This is where the source code lives

import subprocess

# Find the latest stage1 output directory
stage1_dirs = sorted([d for d in INPUTS_DIR.iterdir() if d.is_dir() and d.name.startswith('stage1_')])

if not stage1_dirs:
    print("ERROR: No Stage 1 outputs found in:", INPUTS_DIR)
    print("\nMake sure to:")
    print("1. Run Stage 1 locally: python scripts/orchestrate_pipeline.py --stage1-only")
    print("2. Wait for upload to complete")
    raise FileNotFoundError("Stage 1 outputs not found")

STAGE1_OUTPUT = stage1_dirs[-1]  # Most recent
print(f"Using Stage 1 output from: {STAGE1_OUTPUT}")
print(f"\nContents:")
for artifact in STAGE1_OUTPUT.iterdir():
    if artifact.is_dir():
        file_count = len(list(artifact.rglob('*')))
        print(f"  📁 {artifact.name}: {file_count} files")

In [ ]:
# Clone the project repository you want to document
# This should match what was used in Stage 1

REPO_URL = "https://github.com/AsyncFuncAI/deepwiki-open.git"
REPO_DIR = Path('/tmp/target_repo')

# Clone if not already present
if not REPO_DIR.exists():
    print(f"Cloning repository: {REPO_URL}")
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    print(f"✓ Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")

print(f"\nRepository contents:")
for item in list(REPO_DIR.iterdir())[:10]:
    print(f"  - {item.name}")

## Step 3: Setup Colab Environment and Import Pipeline

In [ ]:
# Clone the GenAI Course project to get the Stage 2 code
COURSE_REPO = "https://github.com/YourUsername/GenAI_Course.git"
COURSE_DIR = Path('/tmp/genai_course')

if not COURSE_DIR.exists():
    print(f"Cloning GenAI Course: {COURSE_REPO}")
    # You'll need to set up your own repo or use a pre-built container
    # For now, create minimal structure
    COURSE_DIR.mkdir(parents=True, exist_ok=True)
    print("Note: Using local Stage 2 code from Drive")

# Add to Python path
import sys
sys.path.insert(0, str(COURSE_DIR))
sys.path.insert(0, '/tmp')

print("✓ Python path configured")

## Step 4: Execute Stage 2 - Documentation Generation

In [ ]:
# Import the pipeline from the cloned repo
# This assumes you have the main.py and supporting modules

# For now, provide template for importing
print("Stage 2 configuration:")
print(f"  Repo root: {REPO_DIR}")
print(f"  Artifacts dir: {STAGE1_OUTPUT}")
print(f"  Output dir: {OUTPUTS_DIR}")
print(f"  GPU available: {torch.cuda.is_available()}")

# If you have the Stage 2 code available, import and execute:
# from main import run_pipeline
# 
# result = run_pipeline(
#     repo_root=str(REPO_DIR),
#     artifacts_dir=str(STAGE1_OUTPUT.parent),  # Parent is /stage2_inputs
#     model_id="HuggingFaceH4/zephyr-7b-alpha",  # or your preferred model
#     device="cuda",
#     dtype="float16",
#     quantize=True,
#     docstring_mode="modules_only",
#     generate_readme=True,
# )
#
# print("Stage 2 completed!")
# print(result)

print("\nTo execute Stage 2:")
print("1. Import the Stage 2 code from your repository")
print("2. Call run_pipeline() with appropriate parameters")
print("3. Results will be saved to OUTPUTS_DIR")

In [ ]:
# PLACEHOLDER FOR ACTUAL STAGE 2 EXECUTION
# Replace with your actual Stage 2 pipeline code

print("Stage 2 execution would happen here.")
print("\nExpected outputs:")
print("  - Generated docstrings")
print("  - Module summaries")
print("  - Architecture documentation")
print("  - README.md")
print("\nThese will be saved to:", OUTPUTS_DIR)

## Step 5: Verify and Upload Results

In [ ]:
# Verify outputs were generated
output_files = list(OUTPUTS_DIR.rglob('*'))
output_file_count = len([f for f in output_files if f.is_file()])

print(f"Output files generated: {output_file_count}")

if output_file_count > 0:
    print(f"\nOutputs directory: {OUTPUTS_DIR}")
    print(f"\nSample outputs:")
    for f in list(output_files)[:20]:
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            print(f"  {f.relative_to(OUTPUTS_DIR)}: {size_kb:.1f} KB")
else:
    print("\n⚠️ No output files generated yet")
    print("Stage 2 execution may have failed")

In [ ]:
# Results are automatically saved to Google Drive (they're already there)
# You can download them locally

from datetime import datetime

# Create summary
summary = {
    "stage": "stage2",
    "timestamp": datetime.now().isoformat(),
    "repo_url": REPO_URL,
    "repo_dir": str(REPO_DIR),
    "stage1_inputs_dir": str(STAGE1_OUTPUT),
    "stage2_outputs_dir": str(OUTPUTS_DIR),
    "output_file_count": output_file_count,
    "gpu_used": torch.cuda.is_available()
}

# Save summary
summary_file = OUTPUTS_DIR / "execution_summary.json"
import json
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("✓ Execution summary saved")
print(json.dumps(summary, indent=2))

## Summary

✅ Stage 2 execution complete!

**Results** are saved on Google Drive at:
```
GenAI_Pipeline_Automation/stage2_outputs/
```

You can now:
1. Download results to your local machine: `python scripts/orchestrate_pipeline.py --download-only`
2. View on Drive: Open Google Drive > GenAI_Pipeline_Automation > stage2_outputs
3. Share or publish the generated documentation

---